In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv
import os
import re
import time

CSV_FILE = "investing.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.investing.com/",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}


def load_existing_urls():
    if not os.path.exists(CSV_FILE):
        return set()
    with open(CSV_FILE, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        return {row["url"] for row in reader}

def get_article_links(base_url):
    try:
        resp = requests.get(base_url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        links = set()
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/news/" in href:
                full_url = urljoin("https://www.investing.com", href)
                links.add(full_url)

        return sorted(links)

    except Exception as e:
        print(f"Error fetching {base_url}: {e}")
        return []

def scrape_article_investing(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        # Title
        title_tag = soup.find("div", id="articleTitle")
        title = title_tag.h1.get_text(strip=True) if title_tag and title_tag.h1 else ""

        # Date
        date_div = soup.find("div", class_=lambda c: c and "flex-col" in c)
        published_date = ""
        if date_div:
            spans = date_div.find_all("span")
            if spans:
                published_date = spans[0].get_text(strip=True)

        # Main content
        content_div = soup.find("div", id="article")
        parts = []
        if content_div:
            for tag in content_div.find_all(["p", "li", "h2"]):
                text = tag.get_text(strip=True)
                if text:
                    parts.append(text)

        return title, published_date, "\n".join(parts)

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "", "", ""

def save_articles(articles):
    file_exists = os.path.exists(CSV_FILE)
    with open(CSV_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["title", "url", "published_date", "content"]
        )
        if not file_exists:
            writer.writeheader()
        for article in articles:
            writer.writerow(article)

if __name__ == "__main__":
    base_urls = [
        "https://www.investing.com/search/?q=crypto&tab=news"
    ]

    existing_urls = load_existing_urls()
    all_links = set()

    for url in base_urls:
        section_links = get_article_links(url)
        print(f"Found {len(section_links)} from {url}")
        all_links.update(section_links)
        time.sleep(5)

    print(f"Total unique candidate articles: {len(all_links)}")

    scraped_data = []
    i = 0
    for link in all_links:
        if link in existing_urls:
            print(f"Skipping (already in CSV): {link}")
            continue
        time.sleep(10)
        print(f"Scraping: {link}")
        title, published_date, content = scrape_article_investing(link)
        if content:
            scraped_data.append({
                "title": title,
                "url": link,
                "published_date": published_date,
                "content": content
            })
        i += 1
        if i >= 3:  # Limit for testing
            break
        
    if scraped_data:
        save_articles(scraped_data)
        print(f"Saved {len(scraped_data)} new articles.")
    else:
        print("No new articles found.")


Found 29 from https://www.investing.com/search/?q=crypto&tab=news
Total unique candidate articles: 29
Scraping: https://www.investing.com/news/stock-market-news/how-consumer-companies-are-using-ai-4168708
Error scraping https://www.investing.com/news/stock-market-news/how-consumer-companies-are-using-ai-4168708: 403 Client Error: Forbidden for url: https://www.investing.com/news/stock-market-news/how-consumer-companies-are-using-ai-4168708
Scraping: https://www.investing.com/news/politics/white-house-crypto-adviser-bo-hines-announces-departure-4181664
Error scraping https://www.investing.com/news/politics/white-house-crypto-adviser-bo-hines-announces-departure-4181664: 403 Client Error: Forbidden for url: https://www.investing.com/news/politics/white-house-crypto-adviser-bo-hines-announces-departure-4181664
Scraping: https://www.investing.com/news/press-releases/the-top-7-mistakes-crypto-investors-make-without-a-cpa-4181613
Error scraping https://www.investing.com/news/press-releases/t